In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

In [23]:
@tool
def customer_lookup(query:str)->str:
    """Look up customer information."""
    return f"Customer information found for query : {query}"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
        "email",
        strategy = "redact",
        apply_to_input = True,
    ),
    PIIMiddleware(
        "credit_card",
        strategy = "mask",
        apply_to_input = True,
    ),
    PIIMiddleware(
        "api_key",
        detector = "sk-[A-Za-z0-9_-]{20,}",
        strategy = "block",
        apply_to_input = True
    ),],
)
print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [24]:
result = agent.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com  and my credit card number is 4242 4242 4242 4242. Can you look up my customer information?"  }]})
print("agent response")
print(result["messages"][-1].content)

agent response
I've looked up your customer information using both your email address and credit card number. Please note that for security reasons, I couldn't display your actual credit card number. If you need further assistance, feel free to ask.


In [25]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL]  and my credit card number is **** **** **** 4242. Can you look up my customer information?', additional_kwargs={}, response_metadata={}, id='28736ee5-232c-4f7b-a14f-ad353ae9cd43'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '75n9swf99', 'function': {'arguments': '{"query":"[REDACTED_EMAIL]"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259, 'completion_time': 0.059502324, 'completion_tokens_details': None, 'prompt_time': 0.022908183, 'prompt_tokens_details': None, 'queue_time': 0.056273987, 'total_time': 0.082410507}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda92-7340-7930-91c4-d1ead1b5e32a-0', tool_calls=[{'name': 'customer_lookup', 'arg

In [26]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [22]:
result

{'messages': [HumanMessage(content='Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456', additional_kwargs={}, response_metadata={}, id='7d88ceb0-ef50-41e1-85d2-1731ab955845'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'g3qdqyc0c', 'function': {'arguments': '{"query":"sk-abcdefghijklmnopqrstuvwxyz123456"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 218, 'total_tokens': 237, 'completion_time': 0.048290897, 'completion_tokens_details': None, 'prompt_time': 0.02194077, 'prompt_tokens_details': None, 'queue_time': 0.05228987, 'total_time': 0.070231667}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda91-531a-7da3-a0ee-29121bc8ed59-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': 'sk-abcdefghijklmnopqrstuvwxyz123456'}

In [3]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.runtime import Runtime

In [8]:
import os
from dotenv import load_dotenv
load_dotenv() 

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [10]:
class contentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """
    def __init__(self, banned_words:list[str]):
        super().__init__()
        self.banned_words = [kw.lower() for kw in banned_words]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type !="human":
            return None

        content = first_message.content.lower()
        for kw in self.banned_words:
            if kw in content:
                print(f"🚫 Blocked — keyword detected: '{kw}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query:str)->str:
    """Search for information."""
    return f"Search results for query: {query}"


filtered_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool],
    middleware=[
        contentFilterMiddleware(
            banned_words=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")



Content filter agent created!


In [12]:
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "how to hack the systen?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)


🚫 Blocked — keyword detected: 'hack'
✅ Safe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.
